# Creating a tiny dataset and writing a loss function

In [8]:
import random
import math

In [9]:
## Import the Value class from lecture 1-d.
class Value:
    def __init__(self, data, _children = (), _op = '', label = ''):
        self.data = data
        self._prev = set(_children)
        self._backward = lambda: None
        self._op = _op
        self.label = label
        self.grad = 0

    def __repr__(self):
        return(f"Value (data = {self.data})")

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)     ## keep other as it is if it is Value object, else wrap it in Value object.
        out = Value(self.data + other.data, _children = (self, other), _op = '+')

        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __radd__(self, other):  ## reverse mul, other * mul
        return self + other
        
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, _children = (self, other), _op = '*')
        
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supporting int/float powers for now"
        out = Value(self.data**other, (self, ), _op = f'**{other}')

        def _backward():
            self.grad += other * self.data**(other - 1) * out.grad

        out._backward = _backward
        return out

    def __rmul__(self, other):  ## reverse mul, other * mul
        return self * other

    def __truediv__(self, other):
        return self * other**-1  ## self/other

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)
    
    # def tanh(self):
    #     out = np.tanh(self.data)
    #     return Value(out, _children = (self, ), _op = 'tanh')

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
        out = Value(t, _children = (self, ), _op = 'tanh')

        def _backward():
            self.grad += (1 - t**2) * out.grad

        out._backward = _backward
        return out

    def exp(self):
        x = self.data
        out = Value(math.exp(x), (self, ), 'exp')

        def _backward():
            self.grad += out.data * out.grad

        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        
        build_topo(self)
        self.grad = 1.0
        
        for node in reversed(topo):
            node._backward() 

## Let's add grad to our visualization pipeline as well

from graphviz import Digraph

def trace(root):
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)

    build(root)
    return nodes, edges

def draw_dot(root):
    dot = Digraph(format = 'svg', graph_attr = {'rankdir': 'LR'}) ## LR - Left to Right
    nodes, edges = trace(root)

    for n in nodes:
        uid = str(id(n))   # The id() function returns a unique id for the specified object.

        # for any value in the graph, create a rectangular ('record') node for it
        dot.node(name = uid, label = "{ %s | data %.4f | grad %.4f}" % (n.label, n.data, n.grad ), shape = 'record')

        if n._op:
            # if this value is a result of some operation, create an op node for it
            dot.node(name = uid + n._op, label = n._op)
            # and connect this node to it
            dot.edge(uid + n._op, uid)

    # if the nodes are in edges
    for n1, n2 in edges:
        # connect n1 to the op node of n2
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)

    return dot
    
a = Value(2.0, label = 'a')
b = Value(-3.0, label = 'b')
c = Value(10.0, label = 'c')
e = a * b   ## python will internally call a.__add__(b)
e.label = 'e'

d = e + c
d.label = 'd'

f = Value(-2.0, label = 'f')
L = d * f
L.label = 'L'


In [10]:
class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1,1))

    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w,x)), self.b)
        out = act.tanh()
        return out

class Layer:
    ## how  many neurons you want in your layer - nout (number of outputs in this layer)
    ## we initialize neurons with this dimesionalaty - (nin)
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)] 

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]  ## outs here is  list, so to make this nicer let's add a check
        return outs[0] if len(outs)==1 else outs

class MLP:
    ## we will take the number of inputs as before - nin
    ## instead of single nout, we will take a list of nout, and size defines all the list of the layers we want in our MLP

    def __init__(self, nin, nouts):
        sz = [nin] + nouts  ## this command concatenates the input with nout list.
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]   ## everytime we create a Layer of (in, out) and keep moving L->R layer by layer.

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    
        

## Let's define a small tiny dataset

In [24]:
## set up inputs
xs = [
    [2.0, 3.0, -1.0], 
    [3.0, -1.0, 0.5], 
    [0.5, 1.0, 1.0], 
    [1.0, 1.0, -1.0],
]

ys = [1.0, -1.0, -1.0, 1.0] ## desired targets

n = MLP(3, [4, 4,1])
ypred = [n(x) for x in xs]

ypred

[Value (data = 0.9253248682530023),
 Value (data = 0.07667316818616764),
 Value (data = 0.937997113735632),
 Value (data = 0.9232797798751001)]

### So, we know we are not close to our actual values, so the loss would be high.

Let's computer sum of squared error. So we need to take the difference between the actual - predicted:


In [25]:
loss = sum([(yout - ygt)**2 for ygt, yout in zip(ys, ypred)])

In [26]:
loss

Value (data = 4.926520291417125)

### Goal: We want the loss to be close to zero, or zero. The greater the loss is the worst the NN is predicting.

In [27]:
loss.backward()

In [28]:
## something magical happended here

n.layers[0].neurons[0].w[0].grad

-1.5111794608741977

### So here the gradient of the weight of this particular neuron of this particular layer is negative, so that means it's influence on the loss is also negative. So slightly increasing this weight of the neuron of the particular layer will make the loss go down and we have this information for every single one of our neurons and all their parameters.

In [32]:
#draw_dot(loss)

### This shows 4 forward passes of a neural net for evry one of the examples and then it has the loss on top, and it ends with the value of the loss.

So all the parameters have gradients, but the gradients on the input data is not so useful for us, because the input data is not something that is changeable, it's given to the problem. But, the some of the parameters for the NN (like ws, and bs), that we would ofcourse like to change.

## Let's add a convenience code in our class so that we can gather up all the parametrs at once

In [35]:
class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1,1))

    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w,x)), self.b)
        out = act.tanh()
        return out

    def parameters(self):
        return self.w + [self.b]  ## returns a list of ws concatenated with list of bs

class Layer:
    ## how  many neurons you want in your layer - nout (number of outputs in this layer)
    ## we initialize neurons with this dimesionalaty - (nin)
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)] 

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]  ## outs here is  list, so to make this nicer let's add a check
        return outs[0] if len(outs)==1 else outs

    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()] 
        # params = []
        # for neuron in self.neuron:
        #     ps = neuron.parameters()
        #     params.extend(ps)   #  the extend() method adds all elements from an iterable (such as a list, tuple, or set) to the end of an existing list
        # return params

class MLP:
    ## we will take the number of inputs as before - nin
    ## instead of single nout, we will take a list of nout, and size defines all the list of the layers we want in our MLP

    def __init__(self, nin, nouts):
        sz = [nin] + nouts  ## this command concatenates the input with nout list.
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]   ## everytime we create a Layer of (in, out) and keep moving L->R layer by layer.

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]
    
        

In [37]:
x = [2.0, 3.0, -1.0]
n = MLP(3, [4, 4, 1])
n(x)

Value (data = -0.7545272127596534)

In [40]:
len(n.parameters())

41

# This MLP has 41 parameters
Counting by hand:

Layer 1 : 3->4: 3 weights + 1 bias for each 4 neurons = 4*4 = 16

Layer 2: 4->4: 4 weights + 1 bias for each 4 neurons = 5*4 = 20

Layer 3: 4->1: 4 weights + 1 bias for 1 output neuron = 5

Sum = 16+20+5 = 41

In [84]:
## set up inputs
xs = [
    [2.0, 3.0, -1.0], 
    [3.0, -1.0, 0.5], 
    [0.5, 1.0, 1.0], 
    [1.0, 1.0, -1.0],
]

ys = [1.0, -1.0, -1.0, 1.0] ## desired targets

n = MLP(3, [4, 4,1])




In [ ]:
ypred = [n(x) for x in xs]
ypred
loss = sum([(yout - ygt)**2 for ygt, yout in zip(ys, ypred)])

loss.backward()


In [44]:
# so let's look at the current value of weight of a neuron and its gradient

n.layers[0].neurons[0].w[0].data

0.5905871424070661

In [45]:
n.layers[0].neurons[0].w[0].grad

0.1510132532596962

In [46]:
## SO, let's change these parameters

## we want to change p.data slightly according to the gradient information
for p in n.parameters():
    p.data += -0.01 * p.grad

## Let's understand this logic here-

The p.grad answers one question: if I increase this parameter slightly, how much does the loss change?

- Positive gradient: increasing p makes the loss go up.
- Negative gradient: increasing p makes the loss go down.

SO we would like to make the loss go down always.
- So, if p.grad is positive, then *-0.01, will make the number negative, and p.data will decrease, so loss will decrease.
- id p.grad is negative,, then *-0.01 will make the number positive and p.data will increase, so loss will decrease.


## Gradient Descent -- three steps (1) forward pass, calculate loss, (2) backward pass, calculate gradients and (3) update the params

In [77]:
i = 20

for k in range(i):
    # forward pass
    ypred = [n(x) for x in xs]
    loss = sum([(yout - ygt)**2 for ygt, yout in zip(ys, ypred)])

    # backward pass
    loss.backward()

    # update 
    for p in n.parameters():
        p.data += -0.01 * p.grad

    print(k, loss.data)

0 9.176435280903878
1 7.423190585733938
2 4.65820628086958
3 2.2375522840604054
4 0.8044816488125416
5 0.5073105116425254
6 0.40486333389939333
7 0.2880622046700567
8 0.1688640261998703
9 0.08583866540632959
10 0.04317844421900832
11 0.02593638804841706
12 0.02248840420530479
13 0.028559797782648503
14 0.04439060858439277
15 0.06999469551178114
16 0.09691664695026758
17 0.10408881658283611
18 0.07954409045220079
19 0.04344582156664791


In [78]:
ypred

[Value (data = 0.8194624529726697),
 Value (data = -0.9847239661941294),
 Value (data = -0.9976258629412982),
 Value (data = 0.8969804778481728)]

# There is a terrible bug in our code, and it's a subtle bug but a very common bug

## We did not did zero grad in backward. So backward() function has +=, so it keeps on accumulating the grad.

In [98]:
i = 20

for k in range(i):
    # forward pass
    ypred = [n(x) for x in xs]
    loss = sum([(yout - ygt)**2 for ygt, yout in zip(ys, ypred)])

    # backward pass
    for p in n.parameters():
        p.grad = 0.0
    loss.backward()

    # update 
    for p in n.parameters():
        p.data += -0.05 * p.grad

    print(k, loss.data)

0 0.012933068506144115
1 0.012727658899768915
2 0.012528324825850678
3 0.012334806770963299
4 0.012146859607522178
5 0.01196425161852189
6 0.011786763600124678
7 0.011614188034973244
8 0.01144632832983342
9 0.011282998111823525
10 0.011124020578064598
11 0.01096922789409739
12 0.010818460636869028
13 0.010671567278499348
14 0.010528403707398482
15 0.010388832783633013
16 0.010252723925727767
17 0.010119952726350937
18 0.009990400594562729
19 0.009863954422518815


In [99]:
ypred

[Value (data = 0.9574751633544237),
 Value (data = -0.959149934898254),
 Value (data = -0.9348557838799978),
 Value (data = 0.9537064153680078)]

# Summary of what we learned

## What are Neural Nets?

- Neural networks are mathematical expressions that take two kinds of input: the data, and the network’s parameters (its weights and biases).
- The forward pass runs the data through this expression to produce predictions.
- A loss function then measures how far those predictions are from the correct targets: the lower the loss, the better the network is doing.
- Backpropagation calculates the gradient of the loss with respect to every parameter, which tells us how each parameter affects the loss.
- Gradient descent then nudges each parameter a small step in the opposite direction of its gradient, which lowers the loss.

Repeating this loop many times (forward pass, zero the gradients, backward pass, update) is how the network learns.